# Airbridges

Airbridges are ground-plane crossovers that hop *over* a CPW trace to tie
the ground plane together and suppress unwanted slotline modes. In Quantum
Metal an airbridge is a first-class `QComponent`: its geometry lives in the
design, so it shows up in `qm.view` and is exported by every renderer — no
GDS-only special case.

This tutorial covers the `Airbridge` component, automatic placement along a
routed CPW with `route_airbridges`, and GDS export.

## Setup

In [ ]:
import qiskit_metal as qm
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.airbridge import Airbridge, route_airbridges

## A single airbridge

The `Airbridge` component draws two base-metal landing pads joined by an
elevated bridge span. The bridge and the pads are emitted on separate layers
(`bridge_layer` / `pad_layer`) so the two-step fabrication is representable.

In [ ]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

Airbridge(design, "ab", options=dict(crossover_length="24um"))
design.rebuild()
qm.view(design)

Key options (all parseable strings, e.g. `'22um'`):

- `crossover_length` — the gap the bridge spans, foot-to-foot. Size it to
  about `cpw_width + 2*cpw_gap` of the CPW you are crossing.
- `bridge_width` — width of the elevated span.
- `pad_width`, `pad_length` — the base landing-pad footprint.
- `bridge_layer`, `pad_layer` — the fabrication layers.

## Auto-placing airbridges along a CPW

`route_airbridges(design, route, ...)` adds `Airbridge` components along a
routed CPW. Placement follows the route's *filleted* centerline (the geometry
that is actually rendered), so every bridge sits on the trace and crosses it
perpendicular to the local direction — through bends as well as straights.
Bridges are spaced by `pitch` and kept `min_spacing` clear of the end pins.
By default each span is sized to the route's `trace_width + 2*trace_gap`.

In [ ]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

OpenToGround(design, "A", options=dict(pos_x="-1mm", orientation="0"))
OpenToGround(design, "B", options=dict(pos_x="1mm", orientation="180"))

cpw = RouteMeander(
    design,
    "cpw",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(component="A", pin="open"),
            end_pin=Dict(component="B", pin="open"),
        ),
        total_length="8mm",
        fillet="90um",
        trace_width="10um",
        trace_gap="6um",
        meander=Dict(spacing="0.3mm"),
    ),
)
design.rebuild()

bridges = route_airbridges(design, cpw, pitch="0.3mm", min_spacing="30um")
design.rebuild()
print(f"placed {len(bridges)} airbridges")
qm.view(design)

`route_airbridges` is idempotent: re-running it (e.g. re-executing this cell,
or after re-routing) clears the airbridges it previously placed under the same
`name` and re-places them. Tune the spacing, or forward extra options to every
bridge via `ab_options`:

In [ ]:
bridges = route_airbridges(
    design, cpw, pitch="0.5mm", min_spacing="30um", ab_options=dict(bridge_width="10um")
)
design.rebuild()
print(f"placed {len(bridges)} airbridges")
qm.view(design)

### Guaranteeing a bridge on every bend

Uniform spacing already covers bends wherever a `pitch` sample happens to land
on a fillet arc, but with a coarse `pitch` a corner can fall between samples.
Set `bridge_at_corners=True` to force one bridge centered on every rounded
bend, in addition to the uniform-pitch bridges. A uniform bridge that would
land within half a pitch of a corner bridge is dropped so the two do not bunch
up.

In [ ]:
bridges = route_airbridges(
    design, cpw, pitch="0.6mm", min_spacing="30um", bridge_at_corners=True
)
design.rebuild()
print(f"placed {len(bridges)} airbridges (one on every bend)")
qm.view(design)

## Export to GDS

Because airbridges are real components, they export like anything else — the
bridge and pad polygons land on their configured GDS layers. Here we export a
straight CPW with airbridges:

In [ ]:
import tempfile
import os

gdesign = designs.DesignPlanar()
gdesign.overwrite_enabled = True
OpenToGround(gdesign, "A", options=dict(pos_x="-0.6mm", orientation="0"))
OpenToGround(gdesign, "B", options=dict(pos_x="0.6mm", orientation="180"))
gcpw = RouteStraight(
    gdesign,
    "cpw",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(component="A", pin="open"),
            end_pin=Dict(component="B", pin="open"),
        ),
        trace_width="10um",
        trace_gap="6um",
    ),
)
gdesign.rebuild()
route_airbridges(gdesign, gcpw, pitch="0.2mm", min_spacing="30um")
gdesign.rebuild()

path = os.path.join(tempfile.mkdtemp(), "airbridge_demo.gds")
gdesign.renderers.gds.export_to_gds(path)
print("wrote", path)

That's the airbridge workflow: place them (individually or automatically),
see them in `qm.view`, and export. Design-of-record and roadmap (including the
experimental 3D/Ansys span) are tracked in issue #1138.